<a href="https://colab.research.google.com/github/ntuzxy/ML/blob/main/Copy_of_quantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
### Ignore this line
! pip install -q tensorflow-model-optimization

     |████████████████████████████████| 238 kB 10.5 MB/s 


In [ ]:
import numpy as np
import tensorflow as tf
import tensorflow
import time
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import re

from tensorflow import keras
from tensorflow.keras import regularizers
from tensorflow.keras.datasets import mnist
from tensorflow.keras import models, layers
from tensorflow.keras.models import Sequential
from tensorflow.keras import optimizers
from tensorflow.keras import losses
from tensorflow_model_optimization.sparsity import keras as sparsity

# Custom Model on Mnist

In [ ]:
#Load dataset as train and test sets
(x_train, y_train), (x_test, y_test) = mnist.load_data()

11490434/11490434 [==============================] - 0s 0us/step


In [ ]:
x_train = x_train.reshape(60000, 784)
x_test = x_test.reshape(10000, 784)
x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
x_train /= 255
x_test /= 255
print(x_train.shape[0], 'train samples')
print(x_test.shape[0], 'test samples')
num_classes = 10
# convert class vectors to binary class matrices
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

60000 train samples
10000 test samples


In [ ]:
from keras.models import Sequential
from keras import models, layers
from keras import regularizers
model = keras.Sequential()
model.add(keras.layers.Dropout(0.2,input_shape=(784,)))
model.add(keras.layers.Dense(1000,
                        kernel_regularizer = regularizers.l2(0.01),
                        activation='relu'))
model.add(keras.layers.Dropout(0.5))
model.add(keras.layers.Dense(1000,
                        kernel_regularizer = regularizers.l2(0.01),
                        activation='relu'))
model.add(keras.layers.Dropout(0.5))
model.add(keras.layers.Dense(10,  activation='softmax'))
#display the model summary
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dropout (Dropout)           (None, 784)               0         
                                                                 
 dense (Dense)               (None, 1000)              785000    
                                                                 
 dropout_1 (Dropout)         (None, 1000)              0         
                                                                 
 dense_1 (Dense)             (None, 1000)              1001000   
                                                                 
 dropout_2 (Dropout)         (None, 1000)              0         
                                                                 
 dense_2 (Dense)             (None, 10)                10010     
                                                                 
Total params: 1,796,010
Trainable params: 1,796,010
Non-

## (A) Post Training Quantization

In [ ]:
model.compile(loss=keras.losses.categorical_crossentropy,
              optimizer='adam',
              metrics=['accuracy'])

In [ ]:
hist = model.fit(x_train, y_train,
                        batch_size=128,
                        epochs=10,
                        verbose=1,
                        validation_data=(x_test,y_test))

Epoch 1/10
469/469 [==============================] - 5s 5ms/step - loss: 2.0437 - accuracy: 0.8558 - val_loss: 0.6841 - val_accuracy: 0.9214
Epoch 2/10
469/469 [==============================] - 2s 4ms/step - loss: 0.7546 - accuracy: 0.8935 - val_loss: 0.6391 - val_accuracy: 0.9310
Epoch 3/10
469/469 [==============================] - 2s 4ms/step - loss: 0.7303 - accuracy: 0.8983 - val_loss: 0.6082 - val_accuracy: 0.9352
Epoch 4/10
469/469 [==============================] - 2s 4ms/step - loss: 0.7096 - accuracy: 0.9020 - val_loss: 0.5738 - val_accuracy: 0.9423
Epoch 5/10
469/469 [==============================] - 2s 4ms/step - loss: 0.6946 - accuracy: 0.9036 - val_loss: 0.5739 - val_accuracy: 0.9422
Epoch 6/10
469/469 [==============================] - 2s 4ms/step - loss: 0.6763 - accuracy: 0.9041 - val_loss: 0.5604 - val_accuracy: 0.9415
Epoch 7/10
469/469 [==============================] - 2s 4ms/step - loss: 0.6618 - accuracy: 0.9070 - val_loss: 0.5344 - val_accuracy: 0.9441
Epoch 

In [ ]:
score = model.evaluate(x_test, y_test, verbose=1)
print("Test loss {:.4f}, accuracy {:.2f}%".format(score[0], score[1] * 100))

313/313 [==============================] - 1s 2ms/step - loss: 0.5138 - accuracy: 0.9495
Test loss 0.5138, accuracy 94.95%


In [ ]:
#Save the entire model in model.h5 file
model.save("model.h5")
print("Saved model to disk")

Saved model to disk


In [ ]:
# a = model.weights
# print(a)

In [ ]:
model = tf.keras.models.load_model('model.h5')
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
#saving converted model in "converted_model.tflite" file
open("converted_model.tflite", "wb").write(tflite_model)

7186080

In [ ]:
model = tf.keras.models.load_model('model.h5')
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant_model = converter.convert()
#saving converted model in "converted_quant_model.tflite" file
open("converted_quant_model.tflite", "wb").write(tflite_quant_model)

1804248

In [ ]:
import os
print("Float model in Mb:", os.path.getsize('converted_model.tflite') / float(2**20))
print("Quantized model in Mb:", os.path.getsize('converted_quant_model.tflite') / float(2**20))
print("Compression ratio:", os.path.getsize('converted_model.tflite')/os.path.getsize('converted_quant_model.tflite'))

Float model in Mb: 6.853179931640625
Quantized model in Mb: 1.7206649780273438
Compression ratio: 3.982867100309935


In [ ]:
# Load TFLite model and allocate tensors.
interpreter = \
tf.lite.Interpreter(model_path="converted_quant_model.tflite")
interpreter.allocate_tensors()
# Get input and output tensors.
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
# Test model on some input data.
input_shape = input_details[0]['shape']
acc=0
for i in range(len(x_test)):
    input_data = x_test[i].reshape(input_shape)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])
    if(np.argmax(output_data) == np.argmax(y_test[i])):
        acc+=1
acc = acc/len(x_test)
print(acc*100)

94.92


## (B) Quantization Aware Training

### Quantize full model

In [ ]:
from keras.models import Sequential
from keras import models, layers
from keras import regularizers
model = keras.Sequential()
model.add(keras.layers.Dropout(0.2,input_shape=(784,)))
model.add(keras.layers.Dense(1000,
                        kernel_regularizer = regularizers.l2(0.01),
                        activation='relu'))
model.add(keras.layers.Dropout(0.5))
model.add(keras.layers.Dense(1000,
                        kernel_regularizer = regularizers.l2(0.01),
                        activation='relu'))
model.add(keras.layers.Dropout(0.5))
model.add(keras.layers.Dense(10,  activation='softmax'))
#display the model summary
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dropout_3 (Dropout)         (None, 784)               0         
                                                                 
 dense_3 (Dense)             (None, 1000)              785000    
                                                                 
 dropout_4 (Dropout)         (None, 1000)              0         
                                                                 
 dense_4 (Dense)             (None, 1000)              1001000   
                                                                 
 dropout_5 (Dropout)         (None, 1000)              0         
                                                                 
 dense_5 (Dense)             (None, 10)                10010     
                                                                 
Total params: 1,796,010
Trainable params: 1,796,010
No

In [ ]:
model.compile(loss=keras.losses.categorical_crossentropy,
              optimizer='adam',
              metrics=['accuracy'])

In [ ]:
hist = model.fit(x_train, y_train,
                        batch_size=128,
                        epochs=100,
                        verbose=1,
                        validation_split = 0.1)

Epoch 1/100
422/422 [==============================] - 2s 4ms/step - loss: 2.2076 - accuracy: 0.8483 - val_loss: 0.6686 - val_accuracy: 0.9370
Epoch 2/100
422/422 [==============================] - 2s 4ms/step - loss: 0.7813 - accuracy: 0.8869 - val_loss: 0.6219 - val_accuracy: 0.9415
Epoch 3/100
422/422 [==============================] - 2s 4ms/step - loss: 0.7387 - accuracy: 0.8965 - val_loss: 0.6137 - val_accuracy: 0.9448
Epoch 4/100
422/422 [==============================] - 3s 6ms/step - loss: 0.7180 - accuracy: 0.8988 - val_loss: 0.5607 - val_accuracy: 0.9502
Epoch 5/100
422/422 [==============================] - 3s 7ms/step - loss: 0.7021 - accuracy: 0.9014 - val_loss: 0.5464 - val_accuracy: 0.9607
Epoch 6/100
422/422 [==============================] - 3s 7ms/step - loss: 0.6940 - accuracy: 0.9017 - val_loss: 0.5288 - val_accuracy: 0.9558
Epoch 7/100
422/422 [==============================] - 3s 6ms/step - loss: 0.6799 - accuracy: 0.9043 - val_loss: 0.5208 - val_accuracy: 0.9542

In [ ]:
import tensorflow_model_optimization as tfmot

quantize_model = tfmot.quantization.keras.quantize_model

# q_aware stands for for quantization aware.
q_aware_model = quantize_model(model)

# `quantize_model` requires a recompile.
q_aware_model.compile(loss=keras.losses.categorical_crossentropy,
              optimizer='adam',
              metrics=['accuracy'])

q_aware_model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 quantize_layer (QuantizeLay  (None, 784)              3         
 er)                                                             
                                                                 
 quant_dropout_3 (QuantizeWr  (None, 784)              1         
 apperV2)                                                        
                                                                 
 quant_dense_3 (QuantizeWrap  (None, 1000)             785005    
 perV2)                                                          
                                                                 
 quant_dropout_4 (QuantizeWr  (None, 1000)             1         
 apperV2)                                                        
                                                                 
 quant_dense_4 (QuantizeWrap  (None, 1000)            

In [ ]:
histq = q_aware_model.fit(x_train, y_train,
                        batch_size=128,
                        epochs=100,
                        verbose=1,
                        validation_split = 0.1)

Epoch 1/100
422/422 [==============================] - 3s 6ms/step - loss: 0.5389 - accuracy: 0.9051 - val_loss: 0.3580 - val_accuracy: 0.9627
Epoch 2/100
422/422 [==============================] - 2s 5ms/step - loss: 0.5373 - accuracy: 0.9081 - val_loss: 0.3695 - val_accuracy: 0.9613
Epoch 3/100
422/422 [==============================] - 2s 5ms/step - loss: 0.5456 - accuracy: 0.9074 - val_loss: 0.3653 - val_accuracy: 0.9667
Epoch 4/100
422/422 [==============================] - 2s 5ms/step - loss: 0.5465 - accuracy: 0.9099 - val_loss: 0.3638 - val_accuracy: 0.9692
Epoch 5/100
422/422 [==============================] - 2s 5ms/step - loss: 0.5458 - accuracy: 0.9093 - val_loss: 0.3750 - val_accuracy: 0.9650
Epoch 6/100
422/422 [==============================] - 2s 5ms/step - loss: 0.5454 - accuracy: 0.9092 - val_loss: 0.3758 - val_accuracy: 0.9647
Epoch 7/100
422/422 [==============================] - 2s 5ms/step - loss: 0.5458 - accuracy: 0.9094 - val_loss: 0.3678 - val_accuracy: 0.9657

In [ ]:
_, baseline_model_accuracy = model.evaluate(
    x_test, y_test, verbose=1)

_, q_aware_model_accuracy = q_aware_model.evaluate(
    x_test, y_test, verbose=1)

print('Baseline test accuracy:', baseline_model_accuracy*100)
print('Quant test accuracy:', q_aware_model_accuracy*100)

313/313 [==============================] - 1s 3ms/step - loss: 0.3858 - accuracy: 0.9541
Baseline test accuracy: 95.55000066757202
Quant test accuracy: 95.41000127792358


In [ ]:
#### fine tune with QAT on a subset of the training data.

In [ ]:
histq= q_aware_model.fit(x_train[:1000], y_train[:1000],
                        batch_size=128,
                        epochs=100,
                        verbose=1,
                        validation_split = 0.1)

Epoch 1/100
8/8 [==============================] - 0s 13ms/step - loss: 0.5160 - accuracy: 0.9111 - val_loss: 0.4375 - val_accuracy: 0.9500
Epoch 2/100
8/8 [==============================] - 0s 9ms/step - loss: 0.5156 - accuracy: 0.9133 - val_loss: 0.4639 - val_accuracy: 0.9500
Epoch 3/100
8/8 [==============================] - 0s 8ms/step - loss: 0.4960 - accuracy: 0.9167 - val_loss: 0.4864 - val_accuracy: 0.9400
Epoch 4/100
8/8 [==============================] - 0s 10ms/step - loss: 0.4499 - accuracy: 0.9322 - val_loss: 0.5587 - val_accuracy: 0.9400
Epoch 5/100
8/8 [==============================] - 0s 9ms/step - loss: 0.4354 - accuracy: 0.9422 - val_loss: 0.5836 - val_accuracy: 0.9400
Epoch 6/100
8/8 [==============================] - 0s 8ms/step - loss: 0.4419 - accuracy: 0.9378 - val_loss: 0.5416 - val_accuracy: 0.9400
Epoch 7/100
8/8 [==============================] - 0s 8ms/step - loss: 0.4225 - accuracy: 0.9511 - val_loss: 0.6709 - val_accuracy: 0.9400
Epoch 8/100
8/8 [========

In [ ]:
_, baseline_model_accuracy = model.evaluate(
    x_test, y_test, verbose=1)

_, q_aware_model_accuracy = q_aware_model.evaluate(
    x_test, y_test, verbose=1)

print('Baseline test accuracy:', baseline_model_accuracy*100)
print('Quant test accuracy:', q_aware_model_accuracy*100)

313/313 [==============================] - 1s 3ms/step - loss: 0.6610 - accuracy: 0.9059
Baseline test accuracy: 95.55000066757202
Quant test accuracy: 90.59000015258789


### Quantize some layers

In [ ]:
import tensorflow_model_optimization as tfmot
annotate = tfmot.quantization.keras.quantize_annotate_layer

In [ ]:
from keras.models import Sequential
from keras import models, layers
from keras import regularizers
model = keras.Sequential()
model.add(keras.layers.Dropout(0.2,input_shape=(784,)))
model.add(keras.layers.Dense(1000,
                        kernel_regularizer = regularizers.l2(0.01),
                        activation='relu'))
model.add(keras.layers.Dropout(0.5))
model.add(annotate(keras.layers.Dense(1000,
                        kernel_regularizer = regularizers.l2(0.01),
                        activation='relu')))
model.add(keras.layers.Dropout(0.5))
model.add(keras.layers.Dense(10,  activation='softmax'))
#display the model summary
model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dropout_6 (Dropout)         (None, 784)               0         
                                                                 
 dense_6 (Dense)             (None, 1000)              785000    
                                                                 
 dropout_7 (Dropout)         (None, 1000)              0         
                                                                 
 quantize_annotate_6 (Quanti  (None, 1000)             1001000   
 zeAnnotate)                                                     
                                                                 
 dropout_8 (Dropout)         (None, 1000)              0         
                                                                 
 dense_8 (Dense)             (None, 10)                10010     
                                                      

In [ ]:
# Use `quantize_apply` to actually make the model quantization aware.
quant_aware_model = tfmot.quantization.keras.quantize_apply(model)

quant_aware_model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dropout_6 (Dropout)         (None, 784)               0         
                                                                 
 dense_6 (Dense)             (None, 1000)              785000    
                                                                 
 quant_dropout_7 (QuantizeWr  (None, 1000)             1         
 apperV2)                                                        
                                                                 
 quant_dense_7 (QuantizeWrap  (None, 1000)             1001005   
 perV2)                                                          
                                                                 
 dropout_8 (Dropout)         (None, 1000)              0         
                                                                 
 dense_8 (Dense)             (None, 10)               

In [ ]:
quant_aware_model.compile(loss=keras.losses.categorical_crossentropy,
              optimizer='adam',
              metrics=['accuracy'])


In [ ]:
histq = quant_aware_model.fit(x_train, y_train,
                        batch_size=128,
                        epochs=100,
                        verbose=1,
                        validation_split = 0.1)

Epoch 1/100
422/422 [==============================] - 3s 5ms/step - loss: 2.1979 - accuracy: 0.8511 - val_loss: 0.6638 - val_accuracy: 0.9335
Epoch 2/100
422/422 [==============================] - 2s 4ms/step - loss: 0.7777 - accuracy: 0.8880 - val_loss: 0.6118 - val_accuracy: 0.9438
Epoch 3/100
422/422 [==============================] - 2s 4ms/step - loss: 0.7430 - accuracy: 0.8963 - val_loss: 0.5810 - val_accuracy: 0.9473
Epoch 4/100
422/422 [==============================] - 2s 4ms/step - loss: 0.7175 - accuracy: 0.9009 - val_loss: 0.5747 - val_accuracy: 0.9490
Epoch 5/100
422/422 [==============================] - 2s 4ms/step - loss: 0.7049 - accuracy: 0.9023 - val_loss: 0.5749 - val_accuracy: 0.9467
Epoch 6/100
422/422 [==============================] - 2s 4ms/step - loss: 0.6919 - accuracy: 0.9048 - val_loss: 0.5579 - val_accuracy: 0.9490
Epoch 7/100
422/422 [==============================] - 2s 4ms/step - loss: 0.6827 - accuracy: 0.9048 - val_loss: 0.5164 - val_accuracy: 0.9582

In [ ]:
_, quant_aware_model_accuracy = quant_aware_model.evaluate(
    x_test, y_test, verbose=1)

print('Quant test accuracy:', quant_aware_model_accuracy*100)

313/313 [==============================] - 1s 3ms/step - loss: 0.3914 - accuracy: 0.9571
Quant test accuracy: 95.7099974155426


# VGG on CIFAR10

In [ ]:
from keras.datasets import cifar10
(x_train,y_train),(x_test,y_test)=cifar10.load_data()

170498071/170498071 [==============================] - 13s 0us/step


In [ ]:
vgg_model = tf.keras.applications.VGG16(input_shape=x_train[0].shape, include_top=False, weights=None)

vgg_model.trainable = True
global_average_layer = tf.keras.layers.GlobalAveragePooling2D()
global_max_layer = tf.keras.layers.GlobalMaxPool2D()

model = keras.Sequential()
model.add(vgg_model)
model.add(global_average_layer)
model.add(keras.layers.Dense(1024, activation='relu'))
# model.add(keras.layers.BatchNormalization())
# model.add(keras.layers.Dropout(0.5))
model.add(keras.layers.Dense(10, activation='softmax'))
print(model.summary())

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 vgg16 (Functional)          (None, 1, 1, 512)         14714688  
                                                                 
 global_average_pooling2d (G  (None, 512)              0         
 lobalAveragePooling2D)                                          
                                                                 
 dense_9 (Dense)             (None, 1024)              525312    
                                                                 
 dense_10 (Dense)            (None, 10)                10250     
                                                                 
Total params: 15,250,250
Trainable params: 15,250,250
Non-trainable params: 0
_________________________________________________________________
None


In [ ]:
from tensorflow.keras import losses
#sgd = optimizers.SGD(lr=0.01, decay=1e-6, momentum=0.9, nesterov=True)
model.compile(optimizer='adam',
              loss=losses.sparse_categorical_crossentropy,
              metrics=["accuracy"])

In [ ]:
hist = model.fit(x_train, y_train,
                        batch_size=128,
                        epochs=20,
                        verbose=1,
                        validation_split=0.1)

Epoch 1/20
352/352 [==============================] - 32s 63ms/step - loss: 2.3985 - accuracy: 0.1002 - val_loss: 2.3029 - val_accuracy: 0.0950
Epoch 2/20
352/352 [==============================] - 20s 58ms/step - loss: 2.3123 - accuracy: 0.1009 - val_loss: 2.3019 - val_accuracy: 0.1064
Epoch 3/20
352/352 [==============================] - 20s 57ms/step - loss: 2.3062 - accuracy: 0.1046 - val_loss: 2.3033 - val_accuracy: 0.0976
Epoch 4/20
352/352 [==============================] - 20s 57ms/step - loss: 2.3045 - accuracy: 0.1071 - val_loss: 2.2847 - val_accuracy: 0.1114
Epoch 5/20
352/352 [==============================] - 20s 58ms/step - loss: 2.1844 - accuracy: 0.1387 - val_loss: 1.8827 - val_accuracy: 0.2240
Epoch 6/20
352/352 [==============================] - 20s 58ms/step - loss: 1.8089 - accuracy: 0.2766 - val_loss: 1.7055 - val_accuracy: 0.3366
Epoch 7/20
352/352 [==============================] - 20s 58ms/step - loss: 1.5970 - accuracy: 0.3850 - val_loss: 1.5182 - val_accuracy:

In [ ]:
score = model.evaluate(x_test, y_test, verbose=1)
print("Test loss {:.4f}, accuracy {:.2f}%".format(score[0], score[1] * 100))

313/313 [==============================] - 4s 10ms/step - loss: 1.0167 - accuracy: 0.7126
Test loss 1.0167, accuracy 71.26%


In [ ]:
model.save("E:/python/tensorflow/TF_Quantization-master/vgg_model.h5")
print("Saved model to disk")

Saved model to disk


In [ ]:
### model = tf.keras.models.load_model('/media/mnt/YONGJUN/TF_Quantization-master/vgg_model.h5')
model = tf.keras.models.load_model('E:/python/tensorflow/TF_Quantization-master/vgg_model.h5')
model.compile(optimizer='adam',
              loss=losses.sparse_categorical_crossentropy,
              metrics=["accuracy"])
hist = model.fit(x_train, y_train,
                        batch_size=128,
                        epochs=50,
                        verbose=1,
                        validation_split=0.1)

Epoch 1/50
352/352 [==============================] - 23s 64ms/step - loss: 0.3274 - accuracy: 0.8914 - val_loss: 1.1051 - val_accuracy: 0.7286
Epoch 2/50
352/352 [==============================] - 21s 61ms/step - loss: 0.2834 - accuracy: 0.9058 - val_loss: 1.0460 - val_accuracy: 0.7296
Epoch 3/50
352/352 [==============================] - 21s 60ms/step - loss: 0.2589 - accuracy: 0.9136 - val_loss: 1.0525 - val_accuracy: 0.7362
Epoch 4/50
352/352 [==============================] - 21s 59ms/step - loss: 0.2388 - accuracy: 0.9216 - val_loss: 1.0855 - val_accuracy: 0.7316
Epoch 5/50
352/352 [==============================] - 21s 59ms/step - loss: 0.2327 - accuracy: 0.9235 - val_loss: 1.1021 - val_accuracy: 0.7412
Epoch 6/50
352/352 [==============================] - 22s 63ms/step - loss: 0.2222 - accuracy: 0.9273 - val_loss: 1.0941 - val_accuracy: 0.7380
Epoch 7/50
352/352 [==============================] - 21s 59ms/step - loss: 0.1941 - accuracy: 0.9379 - val_loss: 1.1365 - val_accuracy:

# Quantization Aware Training

In [ ]:
# Helper function uses `quantize_annotate_layer` to annotate that only the
# Dense layers should be quantized.
def apply_quantization_to_dense(layer):
  if isinstance(layer, tf.keras.layers.Dense):
    return tfmot.quantization.keras.quantize_annotate_layer(layer)
  return layer

model = tf.keras.models.load_model('vgg_model.h5')

# Use `tf.keras.models.clone_model` to apply `apply_quantization_to_dense`
# to the layers of the model.
annotated_model = tf.keras.models.clone_model(
    model,
    clone_function=apply_quantization_to_dense,
)

# https://www.tensorflow.org/model_optimization/api_docs/python/tfmot/quantization/keras/experimental/default_n_bit/DefaultNBitQuantizeScheme
resolution = tfmot.quantization.keras.experimental.default_n_bit.DefaultNBitQuantizeScheme(
    disable_per_axis=False, num_bits_weight=3, num_bits_activation=3
)

# Now that the Dense layers are annotated,
# `quantize_apply` actually makes the model quantization aware.
quant_aware_model = tfmot.quantization.keras.quantize_apply(annotated_model,resolution)
quant_aware_model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])
quant_aware_model.summary()
open("vgg_quant_model.tflite", "wb").write(tflite_quant_model)

## Train and evaluate the quantized model against baseline

In [ ]:
train_images_subset = x_train[0:1000] # out of 50000
train_labels_subset = y_train[0:1000]

quant_aware_model.fit(train_images_subset, train_labels_subset,
                  batch_size=128, epochs=1, validation_split=0.1)

In [ ]:
_, baseline_model_accuracy = model.evaluate(
    x_test, y_test, verbose=0)

_, quant_aware_model_accuracy = quant_aware_model.evaluate(
   x_test, y_test, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)
print('Quant test accuracy:', quant_aware_model_accuracy)

## Create quantized model for TFLite backend

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(quant_aware_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
quantized_tflite_model = converter.convert()
open("quantized_tflite_model.tflite", "wb").write(quantized_tflite_model)

## See persistence of accuracy from TF to TFLite

In [ ]:
import numpy as np

def evaluate_model(interpreter):
  input_index = interpreter.get_input_details()[0]["index"]
  output_index = interpreter.get_output_details()[0]["index"]

  # Run predictions on every image in the "test" dataset.
  prediction_digits = []
  for i, test_image in enumerate(y_test):
    if i % 1000 == 0:
      print('Evaluated on {n} results so far.'.format(n=i))
    # Pre-processing: add batch dimension and convert to float32 to match with
    # the model's input data format.
    test_image = np.expand_dims(test_image, axis=0).astype(np.float32)
    interpreter.set_tensor(input_index, test_image)

    # Run inference.
    interpreter.invoke()

    # Post-processing: remove batch dimension and find the digit with highest
    # probability.
    output = interpreter.tensor(output_index)
    digit = np.argmax(output()[0])
    prediction_digits.append(digit)

  print('\n')
  # Compare prediction results with ground truth labels to calculate accuracy.
  prediction_digits = np.array(prediction_digits)
  accuracy = (prediction_digits == test_labels).mean()
  return accuracy

In [ ]:
interpreter = tf.lite.Interpreter(model_content=quantized_tflite_model)
interpreter.allocate_tensors()

test_accuracy = evaluate_model(interpreter)

print('Quant TFLite test_accuracy:', test_accuracy)
print('Quant TF test accuracy:', quant_aware_model_accuracy)

In [ ]:
x_test = x_test.astype('float32')

# Load TFLite model and allocate tensors.
interpreter = tf.lite.Interpreter(model_path="vgg_quant_model.tflite")
interpreter.allocate_tensors()

# Get input and output tensors.
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Test model on some input data.
input_shape = input_details[0]['shape']
acc=0
for i in range(len(x_test)):
    input_data = x_test[i].reshape(input_shape)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])
    if(np.argmax(output_data) == y_test[i]):
        acc+=1
acc = acc/len(x_test)
print(acc*100)

In [ ]:
import os
import tempfile

# Create float TFLite model.
float_converter = tf.lite.TFLiteConverter.from_keras_model(model)
float_tflite_model = float_converter.convert()

# Measure sizes of models.
_, float_file = tempfile.mkstemp('.tflite')
_, quant_file = tempfile.mkstemp('.tflite')

with open(quant_file, 'wb') as f:
  f.write(tflite_quant_model)

with open(float_file, 'wb') as f:
  f.write(float_tflite_model)

print("Float model in Mb:", os.path.getsize(float_file) / float(2**20))
print("Quantized model in Mb:", os.path.getsize(quant_file) / float(2**20))

## QAT

### Quantize full model

In [ ]:
import tensorflow as tf

In [ ]:
vgg_model = tf.keras.applications.VGG16(input_shape=(32, 32, 3), include_top=False, weights=None)
vgg_model.trainable = True
global_average_layer = tf.keras.layers.GlobalAveragePooling2D()

#https://github.com/tensorflow/model-optimization/issues/40
model = tf.keras.Sequential()
#Had to add input layer, otherwise ran into trouble when printing model summary as well as converting to tf-lite
model.add(layers.Input(shape=(32, 32, 3)))
input_layer = True
for layer in vgg_model.layers:
    model.add(layer)
model.add(layers.Flatten())
model.add(layers.Dense(1024, activation='relu'))
#model.add(layers.BatchNormalization())
#model.add(layers.Dropout(0.5))
model.add(layers.Dense(10, activation='softmax'))
print(model.summary())

In [ ]:
model.compile(loss=losses.sparse_categorical_crossentropy,
              optimizer='adam',
              metrics=['accuracy'])

hist= model.fit(x_train, y_train,
                        batch_size=128,
                        epochs=20,
                        verbose=1,
                        validation_split = 0.1)

In [ ]:
model.save("vgg_true_model.h5")
print("Saved model to disk")

In [ ]:
import tensorflow_model_optimization as tfmot
#model = tf.keras.models.load_model('vgg_model.h5')
quantize_model = tfmot.quantization.keras.quantize_model

# q_aware stands for for quantization aware.
q_aware_model = quantize_model(model)

# `quantize_model` requires a recompile.
q_aware_model.compile(loss=losses.sparse_categorical_crossentropy,
              optimizer='adam',
              metrics=['accuracy'])

model.compile(loss=losses.sparse_categorical_crossentropy,
              optimizer='adam',
              metrics=['accuracy'])

q_aware_model.summary()

In [ ]:
histq= q_aware_model.fit(x_train[:1000], y_train[:1000],
                        batch_size=128,
                        epochs=10,
                        verbose=1,
                        validation_split = 0.1)

In [ ]:
_, baseline_model_accuracy = model.evaluate(
    x_test, y_test, verbose=1)

_, q_aware_model_accuracy = q_aware_model.evaluate(
    x_test, y_test, verbose=1)

print('Baseline test accuracy:', baseline_model_accuracy*100)
print('Quant test accuracy:', q_aware_model_accuracy*100)

In [ ]:
q_aware_model.save("vgg_qat_full_model.h5")
print("Saved model to disk")


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(q_aware_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant_model = converter.convert()
open("vgg_qat_full_model.tflite", "wb").write(tflite_quant_model)

In [ ]:
print("Float complete model in Mb:", os.stat('vgg_true_model.h5').st_size/ float(2**20))
print("QAT model in Mb:", os.stat('vgg_qat_full_model.h5').st_size/ float(2**20))
print("QAT compressed model in Mb:", os.stat('vgg_qat_full_model.tflite').st_size/ float(2**20))

In [ ]:
import os
import tempfile

# Create float TFLite model.
float_converter = tf.lite.TFLiteConverter.from_keras_model(model)
float_tflite_model = float_converter.convert()

# Measure sizes of models.
_, float_file = tempfile.mkstemp('.tflite')
_, quant_file = tempfile.mkstemp('.tflite')

with open(quant_file, 'wb') as f:
  f.write(tflite_quant_model)

with open(float_file, 'wb') as f:
  f.write(float_tflite_model)

print("Float tflite model in Mb:", os.path.getsize(float_file) / float(2**20))
print("Quantized model in Mb:", os.path.getsize(quant_file) / float(2**20))

In [ ]:
x_test = x_test.astype('float32')

# Load TFLite model and allocate tensors.
interpreter = tf.lite.Interpreter(model_path="vgg_qat_full_model.tflite")
interpreter.allocate_tensors()

# Get input and output tensors.
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Test model on some input data.
input_shape = input_details[0]['shape']
acc=0
for i in range(len(x_test)):
    input_data = x_test[i].reshape(input_shape)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])
    if(np.argmax(output_data) == y_test[i]):
        acc+=1
acc = acc/len(x_test)
print(acc*100)

### Quantize some layer

In [ ]:
annotate = tfmot.quantization.keras.quantize_annotate_layer

In [ ]:
vgg_model = tf.keras.applications.VGG16(input_shape=(32, 32, 3), include_top=False, weights=None)
vgg_model.trainable = True
global_average_layer = tf.keras.layers.GlobalAveragePooling2D()

#https://github.com/tensorflow/model-optimization/issues/40
model = tf.keras.Sequential()
#Had to add input layer, otherwise ran into trouble when printing model summary as well as converting to tf-lite
model.add(layers.Input(shape=(32, 32, 3)))
input_layer = True
for layer in vgg_model.layers:
    model.add(layer)
model.add(layers.Flatten())
model.add(layers.Dense(1024, activation='relu'))
#model.add(layers.BatchNormalization())
model.add(layers.Dropout(0.5))
model.add(annotate(layers.Dense(10, activation='softmax')))
print(model.summary())

In [ ]:
# Use `quantize_apply` to actually make the model quantization aware.
quant_aware_model = tfmot.quantization.keras.quantize_apply(model)

quant_aware_model.summary()

In [ ]:
histq= q_aware_model.fit(x_train[:1000], y_train[:1000],
                        batch_size=128,
                        epochs=10,
                        verbose=1,
                        validation_split = 0.1)

In [ ]:
_, baseline_model_accuracy = model.evaluate(
    x_test, y_test, verbose=1)

_, q_aware_model_accuracy = q_aware_model.evaluate(
    x_test, y_test, verbose=1)

print('Baseline test accuracy:', baseline_model_accuracy*100)
print('Quant test accuracy:', q_aware_model_accuracy*100)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(q_aware_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant_model = converter.convert()
open("vgg_qat_model.tflite", "wb").write(tflite_quant_model)

In [ ]:
import os
import tempfile

# Create float TFLite model.
float_converter = tf.lite.TFLiteConverter.from_keras_model(model)
float_tflite_model = float_converter.convert()

# Measure sizes of models.
_, float_file = tempfile.mkstemp('.tflite')
_, quant_file = tempfile.mkstemp('.tflite')

with open(quant_file, 'wb') as f:
  f.write(tflite_quant_model)

with open(float_file, 'wb') as f:
  f.write(float_tflite_model)

print("Float model in Mb:", os.path.getsize(float_file) / float(2**20))
print("Quantized model in Mb:", os.path.getsize(quant_file) / float(2**20))

In [ ]:
print("Float complete model in Mb:", os.stat('vgg_model.h5').st_size/ float(2**20))
print("QAT compressed model in Mb:", os.stat('vgg_qat_model.tflite').st_size/ float(2**20))

In [ ]:
x_test = x_test.astype('float32')

# Load TFLite model and allocate tensors.
interpreter = tf.lite.Interpreter(model_path="vgg_qat_model.tflite")
interpreter.allocate_tensors()

# Get input and output tensors.
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Test model on some input data.
input_shape = input_details[0]['shape']
acc=0
for i in range(len(x_test)):
    input_data = x_test[i].reshape(input_shape)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])
    if(np.argmax(output_data) == y_test[i]):
        acc+=1
acc = acc/len(x_test)
print(acc*100)

In [ ]:
model = model.weights
print(model)